- **Import dependencies**

In [6]:
%reset -f
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

- **Load the data**

In [74]:
data = pd.read_parquet("/Users/euanbronsky/Downloads/spx_cleaned_final.parquet")

In [75]:
data['K/S'] = data['strike'] / data['underlying_last']
data['rel_spread'] = (data['ask'] - data['bid']) / data['mid']

- **Compute summary statistics**

In [76]:
# Compute summary statistics
statistics = data.groupby(['moneyness_group', 'maturity_group'])[['mid', 'iv_fun', 'delta_fun', 'K/S', 'dte', ]].agg(['mean', 'std'])

# Reshape the format and print
table = statistics.stack(0).unstack(1).swaplevel(0,1, axis=1).reindex(['7-45', '45-90', '90-180', '180-360'],axis=1, level=0).reindex(['DOTM_put', 'OTM_put', 'ATM_put', 'ATM_call', 'OTM_call', 'DOTM_call'], axis=0, level=0)
table

maturity_group                  7-45                 45-90             \
                                mean        std       mean        std   
moneyness_group                                                         
DOTM_put        mid         3.750367   4.006665   6.129305   6.732437   
                iv_fun      0.321310   0.157991   0.364385   0.157530   
                delta_fun  -0.041270   0.035174  -0.038360   0.035366   
                K/S         0.854707   0.095583   0.744116   0.134537   
                dte        24.080467  10.428829  65.292767  12.742178   
OTM_put         mid        21.358800  14.927408  34.004869  22.270118   
                iv_fun      0.185552   0.078631   0.198116   0.065295   
                delta_fun  -0.233938   0.072151  -0.235404   0.072456   
                K/S         0.966999   0.020228   0.941755   0.029815   
                dte        23.812210  10.466422  65.116327  12.684634   
ATM_put         mid        43.467157  25.845685  64.925494  37.185711   
                iv_fun      0.159145   0.068050   0.161650   0.053457   
                delta_fun  -0.435790   0.036073  -0.435841   0.036002   
                K/S         0.994636   0.004810   0.990682   0.007817   
                dte        23.956797  10.479969  65.171382  12.622798   
ATM_call        mid        37.148899  21.820352  52.436482  29.414305   
                iv_fun      0.145711   0.058043   0.147476   0.045332   
                delta_fun   0.435126   0.035947   0.434858   0.035909   
                K/S         1.006680   0.005132   1.010928   0.008256   
                dte        24.152978  10.552048  65.368506  12.600815   
OTM_call        mid        14.990851  11.288307  21.652466  15.616276   
                iv_fun      0.132101   0.054005   0.129825   0.040823   
                delta_fun   0.236187   0.072580   0.236269   0.072508   
                K/S         1.024700   0.013995   1.041373   0.020109   
                dte        23.958790  10.670734  65.251625  12.586637   
DOTM_call       mid         2.261739   2.177613   3.345828   3.336165   
                iv_fun      0.127299   0.053238   0.124573   0.040598   
                delta_fun   0.053948   0.032464   0.052183   0.034176   
                K/S         1.056254   0.033510   1.098486   0.051742   
                dte        23.939552  10.675525  65.185335  12.589397   

maturity_group                 90-180                180-360             
                                 mean        std        mean        std  
moneyness_group                                                          
DOTM_put        mid         10.788576  11.482760   17.466735  17.554546  
                iv_fun       0.382971   0.151711    0.373031   0.131488  
                delta_fun   -0.039256   0.035024   -0.042537   0.035640  
                K/S          0.657910   0.156826    0.584081   0.164421  
                dte        132.751500  26.293217  267.974038  52.224675  
OTM_put         mid         60.583431  37.152873   93.840616  54.782026  
                iv_fun       0.222012   0.062526    0.233381   0.052848  
                delta_fun   -0.231968   0.072159   -0.231730   0.072129  
                K/S          0.912105   0.042940    0.878531   0.056593  
                dte        132.689608  26.778253  271.499461  52.332339  
ATM_put         mid        117.689419  60.755124  177.496203  84.570275  
                iv_fun       0.179062   0.050168    0.189134   0.042370  
                delta_fun   -0.435729   0.036092   -0.433514   0.035366  
                K/S          0.989412   0.012905    0.987442   0.018957  
                dte        131.909693  26.828052  270.229055  52.270836  
ATM_call        mid         92.124813  46.853638  142.956692  66.565688  
                iv_fun       0.164434   0.044353    0.177200   0.038132  
                delta_fun    0.435811   0.035932    0.437183   0.036147  
                K/S          1.02028

In [8]:
# Select one specific date for plotting
one_day = df_selected[df_selected['quote_date'] == '2015-02-27']

# Create maps to convert to values
m_map = {
    'DOTM_put': 1,
    'OTM_put': 2,
    'ATM_put': 3,
    'ATM_call': 4,
    'OTM_call': 5,
    'DOTM_call': 6}
t_map = {
    '7-45': 1,
    '45-90': 2,
    '90-180': 3,
    '180-360': 4}

# Add columns with numerical values
one_day['x_num'] = one_day['moneyness_group'].map(m_map)
one_day['y_num'] = one_day['maturity_group'].map(t_map)

# Plotting environment
fig = plt.figure(figsize=(16,10))
ax = fig.add_subplot(projection='3d')

# Define x, y, z variables
X = one_day['x_num']
Y = one_day['y_num']
Z = one_day['iv']

# Plot the results
surf = ax.plot_trisurf(X, Y, Z,
                       cmap='viridis',
                       linewidth=0.1,
                       antialiased=True,
                       alpha=0.8)

# Plot colorbar
cbar = fig.colorbar(surf, ax=ax, shrink=0.5, aspect=5, pad=0.1)
cbar.set_label('Implied Volatility', rotation=270,labelpad=15)

# Set x and y ticks and labels
ax.set_xticks(list(m_map.values()))
ax.set_xticklabels(list(m_map.keys()))
ax.set_yticks(list(t_map.values()))
ax.set_yticklabels(list(t_map.keys()))

# Set labels
ax.set_ylabel('Days-to-expiration', labelpad=14, fontsize=11)
ax.set_xlabel('Moneyness Group', labelpad=14, fontsize=11)
ax.set_zlabel('Implied Volatility', labelpad=10, fontsize=11)

# Limit the z-axis
ax.set_zlim(0, 0.5)

# Plot the results
plt.tight_layout()
plt.show()

NameError: name 'df_selected' is not defined

In [ ]:
# Extract two specific dates
date_2020 = df_buckets[(df_buckets["quote_date"] == "2020-03-04") & (df_buckets["delta_fun"] > -0.5) & (df_buckets['option_type'] == 'put')]
date_2016 = df_buckets[(df_buckets["quote_date"] == "2016-03-21") & (df_buckets["delta_fun"] > -0.5) & (df_buckets['option_type'] == 'put')]

# Extract relevant 2016 variables
deltas_2020 = date_2020['delta_fun']
ivs_2020 = date_2020['iv']
dtes_2020 = date_2020['dte']

# Extract relevant 2020 variables
deltas_2016 = date_2016['delta_fun']
ivs_2016 = date_2016['iv']
dtes_2016 = date_2016['dte']

In [ ]:
#%matplotlib notebook
#%matplotlib inline

# Set up figure environment
fig, axes = plt.subplots(1,2,figsize=(25,10), subplot_kw={'projection':'3d'})
fig.subplots_adjust(wspace=3)

# 2016 plot
surf1 = axes[0].plot_trisurf(deltas_2016, dtes_2016, ivs_2016,
                       cmap='viridis',
                       linewidth=0.1,
                       antialiased=True,
                       alpha=0.9)

# 2020 plot
surf2 = axes[1].plot_trisurf(deltas_2020, dtes_2020, ivs_2020,
                       cmap='viridis',
                       linewidth=0.1,
                       antialiased=True,
                       alpha=0.9)

# Set x-axis limits
axes[0].set_zlim(0, 0.65)

# Set common specifications
for ax in axes.flat:
    ax.set_xlabel(r'$\Delta$-Moneyness', labelpad=10, fontsize=12)
    ax.set_ylabel('Days to expiration', labelpad=10, fontsize=12)
    ax.set_zlabel('Implied Volatility', labelpad=10, fontsize=12)
    ax.set_xlim(0, -0.5)
    #ax.invert_xaxis()
    ax.view_init(elev=20, azim=-25)

# Plot the colorbars
fig.colorbar(surf1, ax=axes[0], shrink=0.25, aspect=5, pad=0.1)
fig.colorbar(surf2, ax=axes[1], shrink=0.25, aspect=5, pad=0.1)

# Plot the results
plt.tight_layout()
#plt.savefig("/Users/euanbronsky/Downloads/Initial.IV.Plot.png", bbox_inches='tight')
plt.show()